In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('Final_Dataset_AIBD.csv')

C:\Users\apkar\AppData\Local\Temp\ipykernel_20712\639219182.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('Final_Dataset_AIBD.csv')


In [3]:
data

,Unnamed: 0.1,Unnamed: 0,Age,Gender,ICU,RUNDC3A,DEPDC1,BICDL1,ATP5PB,COTL1,...,CCR1,DMTN,CAVIN2,IZUMO4,SEPTIN10,ZCWPW1,DENND4C,TBX3,SCD5,H2AP
0,0,C1,39,male,no,0.410000,0.290000,3.500000,83.130000,793.950000,...,464.730000,24.290000,55.980000,1.790000,1.930000,2.260000,10.800000,0.000000,0.890000,0.0
1,1,C2,63,male,no,0.380000,0.930000,4.370000,94.440000,990.270000,...,177.660000,35.440000,99.140000,1.740000,1.690000,2.610000,6.190000,0.000000,1.130000,0.0
2,2,C3,33,male,no,0.390000,0.100000,2.240000,41.240000,719.210000,...,393.770000,6.810000,12.310000,1.860000,0.380000,6.120000,3.590000,0.000000,0.620000,0.0
3,3,C4,49,male,no,0.560000,0.190000,7.870000,123.220000,804.480000,...,256.230000,24.500000,122.510000,3.430000,1.820000,5.010000,9.480000,0.010000,1.590000,0.0
4,4,C5,49,male,no,0.450000,0.440000,1.920000,72.720000,505.980000,...,482.630000,15.100000,44.620000,0.510000,0.210000,3.150000,4.390000,0.000000,0.650000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180,180,NC81,18,male,NaN,2.535697,0.023387,1.061867,68.486990,878.511376,...,7.195875,18.339219,6.031346,4.281002,0.410692,2.368973,0.392739,0.000254,0.091454,0.0
181,181,NC82,15,male,NaN,0.890256,0.032683,7.413428,65.070772,1311.710166,...,71.612906,13.833811,29.663119,1.689109,0.852816,4.191334,2.895935,0.000254,0.917859,0.0
182,182,NC83,16,female,NaN,1.307012,0.023387,3.190243,50.016863,1346.659513,...,95.839170,19.880872,13.036078,1.895986,1.193758,4.573771,1.575842,0.000254,0.158958,0.0
183,183,NC84,14,male,NaN,1.138351,0.106631,9.613936,52.493368,1060.417768,...,41.763646,14.206949,25.061025,1.858951,0.404773,5.072356,2.224795,0.000254,0.500736,0.0


In [4]:
category = [i[0] for i in data['Unnamed: 0']]
print(category.count('N'))
print(category.count('C'))


85
100


In [5]:
# ---- Normalization: clip negatives to 0, then log2(x + 1) ----
gene_cols = list(data.columns[data.columns.get_loc('ICU') + 1:])
raw = data[gene_cols].apply(pd.to_numeric, errors='coerce')
print(f"Negative values clipped: {(raw < 0).sum().sum()}")
expr = np.log2(raw.clip(lower=0) + 1)
print(f"Raw range: {raw.min().min():.2f} to {raw.max().max():.2f}")
print(f"Normalized range: {expr.min().min():.2f} to {expr.max().max():.2f}")


Negative values clipped: 26883
Raw range: -639.90 to 326585.43
Normalized range: 0.00 to 18.32


In [6]:
from scipy import stats
from statsmodels.stats.multitest import multipletests

ALPHA = 0.05  # 95% significance level (FDR)

# Gene columns start after the metadata columns (..., Age, Gender, ICU)
gene_cols = list(data.columns[data.columns.get_loc('ICU') + 1:])

# Group labels: 'C...' = case (C), 'N...' = control (N)
grp = data['Unnamed: 0'].astype(str).str[0]
case, ctrl = expr[grp == 'C'], expr[grp == 'N']

# Welch's t-test for every gene (two-sided)
t_stat, p_val = stats.ttest_ind(case, ctrl, equal_var=False, nan_policy='omit')

res = pd.DataFrame({
    'gene': gene_cols,
    'mean_case': case.mean().values,
    'mean_ctrl': ctrl.mean().values,
    't_stat': t_stat,
    'p_value': p_val,
})
res['log2_fc'] = res['mean_case'] - res['mean_ctrl']  # difference of means on log2 scale = log2 fold change

# Benjamini-Hochberg FDR correction (genes with NaN p-values, e.g. constant genes, are excluded)
valid = res['p_value'].notna()
res['p_adj'] = np.nan
res.loc[valid, 'p_adj'] = multipletests(res.loc[valid, 'p_value'], alpha=ALPHA, method='fdr_bh')[1]
res['significant'] = res['p_adj'] < ALPHA
res['direction'] = np.where(~res['significant'], 'ns',
                            np.where(res['mean_case'] > res['mean_ctrl'], 'elevated', 'suppressed'))

n_tested = valid.sum()
n_sig = res['significant'].sum()
n_up = (res['direction'] == 'elevated').sum()
n_down = (res['direction'] == 'suppressed').sum()

print(f"Genes tested: {n_tested}")
print(f"Significant genes (BH-FDR < {ALPHA}): {n_sig} ({n_sig / n_tested * 100:.2f}%)")
print(f"  Elevated   : {n_up} ({n_up / n_tested * 100:.2f}%)")
print(f"  Suppressed : {n_down} ({n_down / n_tested * 100:.2f}%)")

sig = res[res['significant']]
print("\nMedian log2 fold change among significant genes:")
print(sig.groupby('direction')['log2_fc'].median())
sig.sort_values('p_adj').head(10)

Genes tested: 17721
Significant genes (BH-FDR < 0.05): 11576 (65.32%)
  Elevated   : 9752 (55.03%)
  Suppressed : 1824 (10.29%)

Median log2 fold change among significant genes:
direction
elevated      0.356535
suppressed   -0.273646
Name: log2_fc, dtype: float64


,gene,mean_case,mean_ctrl,t_stat,p_value,log2_fc,p_adj,significant,direction
17506,TYMS,3.494570,1.211976,17.975190,9.984781e-41,2.282593,1.769403e-36,True,elevated
1681,RRM2,3.477786,1.283717,18.971591,2.986612e-40,2.194069,2.646288e-36,True,elevated
16541,CDK1,2.580084,0.790357,17.468701,1.285810e-37,1.789727,7.595278e-34,True,elevated
5014,NUSAP1,2.859450,1.324342,17.249817,3.139311e-37,1.535108,1.142898e-33,True,elevated
2165,PCLAF,2.748387,1.325940,17.578188,3.224700e-37,1.422447,1.142898e-33,True,elevated
17562,CCNA2,2.150122,0.808233,17.845634,6.919495e-37,1.341889,2.043673e-33,True,elevated
9973,UBE2C,3.435466,2.113614,16.007424,3.617374e-36,1.321852,9.157641e-33,True,elevated
9064,ZWINT,2.760250,1.117050,16.745665,1.533929e-35,1.643200,3.397843e-32,True,elevated
14227,CCNB1,2.267451,0.927788,16.887627,2.086770e-35,1.339663,4.108851e-32,True,elevated
4247,RNA5S2,3.448944,5.743581,-15.409090,8.596164e-35,-2.294637,5.858947e-32,True,suppressed


In [7]:
# ---- Median comparison: Mann-Whitney U test + Benjamini-Hochberg (FDR 5%) ----
# Re-uses `case`, `ctrl`, `gene_cols`, `ALPHA` from the previous cell
u_stat, p_mw = stats.mannwhitneyu(case, ctrl, alternative='two-sided', nan_policy='omit')

res_med = pd.DataFrame({
    'gene': gene_cols,
    'median_case': case.median().values,
    'median_ctrl': ctrl.median().values,
    'U_stat': u_stat,
    'p_value': p_mw,
})
res_med['log2_fc'] = res_med['median_case'] - res_med['median_ctrl']  # log2 scale

valid_m = res_med['p_value'].notna()
res_med['p_adj'] = np.nan
res_med.loc[valid_m, 'p_adj'] = multipletests(res_med.loc[valid_m, 'p_value'],
                                              alpha=ALPHA, method='fdr_bh')[1]
res_med['significant'] = res_med['p_adj'] < ALPHA
res_med['direction'] = np.where(~res_med['significant'], 'ns',
                                np.where(res_med['median_case'] > res_med['median_ctrl'],
                                         'elevated', 'suppressed'))

n_tested_m = valid_m.sum()
n_sig_m = res_med['significant'].sum()
n_up_m = (res_med['direction'] == 'elevated').sum()
n_down_m = (res_med['direction'] == 'suppressed').sum()

print(f"Genes tested: {n_tested_m}")
print(f"Significant genes (Mann-Whitney U, BH-FDR < {ALPHA}): {n_sig_m} ({n_sig_m / n_tested_m * 100:.2f}%)")
print(f"  Elevated   : {n_up_m} ({n_up_m / n_tested_m * 100:.2f}%)")
print(f"  Suppressed : {n_down_m} ({n_down_m / n_tested_m * 100:.2f}%)")

sig_m = res_med[res_med['significant']]
print("\nMedian log2 fold change among significant genes:")
print(sig_m.groupby('direction')['log2_fc'].median())
sig_m.sort_values('p_adj').head(10)

Genes tested: 17721
Significant genes (Mann-Whitney U, BH-FDR < 0.05): 12720 (71.78%)
  Elevated   : 9005 (50.82%)
  Suppressed : 3715 (20.96%)

Median log2 fold change among significant genes:
direction
elevated      0.398047
suppressed   -0.020194
Name: log2_fc, dtype: float64


,gene,median_case,median_ctrl,U_stat,p_value,log2_fc,p_adj,significant,direction
8605,CDC6,1.217227,0.277666,8307.5,5.262420e-29,0.939562,9.325535e-25,True,elevated
2341,ESCO2,0.704872,0.124328,8161.0,4.328707e-27,0.580544,9.843728e-24,True,elevated
14227,CCNB1,2.277834,0.877991,8168.0,3.721703e-27,1.399843,9.843728e-24,True,elevated
17043,MCM10,0.575312,0.065886,8187.0,2.076013e-27,0.509427,9.843728e-24,True,elevated
12025,MELK,1.160271,0.197791,8183.0,2.260696e-27,0.962480,9.843728e-24,True,elevated
14916,PBK,1.017918,0.082806,8154.5,3.737859e-27,0.935111,9.843728e-24,True,elevated
17562,CCNA2,2.116008,0.752299,8201.0,1.376025e-27,1.363709,9.843728e-24,True,elevated
2723,DEPDC1B,1.150560,0.183227,8160.5,4.443870e-27,0.967333,9.843728e-24,True,elevated
5237,DTL,1.025024,0.148263,8152.5,5.853308e-27,0.876762,1.152516e-23,True,elevated
6698,TOP2A,1.803203,0.344761,8136.5,9.530528e-27,1.458442,1.594021e-23,True,elevated


In [8]:
import json

def export_genes(df, path):
    out = {
        'elevation': df.loc[df['direction'] == 'elevated', 'gene'].tolist(),
        'supression': df.loc[df['direction'] == 'suppressed', 'gene'].tolist(),
    }
    with open(path, 'w') as f:
        json.dump(out, f, indent=2)
    print(f"{path}: {len(out['elevation'])} elevation, {len(out['supression'])} supression")

export_genes(res, 'mean_significant_genes.json')
export_genes(res_med, 'median_significant_genes.json')

mean_significant_genes.json: 9752 elevation, 1824 supression
median_significant_genes.json: 9005 elevation, 3715 supression
